# 🔬 LAB 1 — OpenSearch Local com Docker Desktop + Túnel ngrok → Google Colab

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duração estimada:** 50 minutos  
**Ambiente:** Docker Desktop na sua máquina local + Google Colab

---

## 🎯 Objetivo
1. Instalar o **Docker Desktop** localmente (Windows / macOS / Linux)
2. Subir o **OpenSearch 3.x** e **Dashboards** via `docker compose`
3. Expor a porta do OpenSearch para a internet usando o **plugin ngrok** do Docker Desktop
4. Conectar o Google Colab ao OpenSearch local via URL pública do ngrok
5. Validar a comunicação com criação de índice e buscas a partir do Colab

## 🏗️ Arquitetura do Lab
```
┌─────────────────────────────────┐        ┌──────────────────────┐
│  SUA MÁQUINA LOCAL              │        │  GOOGLE COLAB        │
│                                 │        │  (VM na nuvem)       │
│  Docker Desktop                 │        │                      │
│  ┌──────────────────────────┐   │  HTTPS │  ┌────────────────┐  │
│  │  opensearch-node1:9200   │◄──┼────────┼──│  Python Client │  │
│  │  opensearch-dashboards   │   │ ngrok  │  │  opensearch-py │  │
│  │  :5601                   │   │ tunnel │  └────────────────┘  │
│  └──────────────────────────┘   │        │                      │
│         ↑                       │        └──────────────────────┘
│  ngrok plugin (Docker Desktop)  │
│  → URL pública HTTPS            │
└─────────────────────────────────┘
```

> **💡 Por que essa arquitetura?** O OpenSearch roda na sua máquina (dados não saem da rede local), enquanto o Colab usa apenas a URL pública como ponto de entrada. Nos labs de produção (Aula 12), o OpenSearch estará num cluster Kubernetes — o fluxo de conexão remota é idêntico.

---

## 📋 PRÉ-REQUISITOS (execute na sua máquina ANTES de abrir o Colab)

### ✅ Checklist Pré-Lab
- [ ] Docker Desktop instalado e em execução (seção 1 do Roteiro de Instalação)
- [ ] Plugin ngrok instalado no Docker Desktop (seção 3 do Roteiro)
- [ ] Conta gratuita criada em https://ngrok.com e token configurado
- [ ] Stack OpenSearch subida com `docker compose up -d`
- [ ] Tunnel ngrok ativo na porta 9200 → URL pública anotada

> **🖥️ Este notebook roda 100% no Google Colab.** Toda a infraestrutura (OpenSearch, ngrok) está na sua máquina local.

---
## 🖥️ PARTE 1 — CONFIGURAÇÃO LOCAL (execute na sua máquina)

### Passo A — Criar o arquivo `docker-compose.yml`

Na sua máquina local, abra um terminal e execute:

```bash
# Cria o diretório do projeto
mkdir -p ~/mba-rag/infra
cd ~/mba-rag/infra

# Windows (PowerShell): use New-Item -ItemType Directory -Path $HOME\mba-rag\infra
```

Crie o arquivo `docker-compose.yml` com o conteúdo abaixo:

```yaml
version: '3'
services:
  opensearch-node1:
    image: opensearchproject/opensearch:3.0.0
    container_name: opensearch-node1
    environment:
      - cluster.name=opensearch-cluster-rag
      - node.name=opensearch-node1
      - discovery.type=single-node
      - bootstrap.memory_lock=true
      - OPENSEARCH_JAVA_OPTS=-Xms6g -Xmx6g
      - plugins.security.disabled=true   # APENAS DESENVOLVIMENTO
      - OPENSEARCH_INITIAL_ADMIN_PASSWORD=Admin@1234!
    ulimits:
      memlock:
        soft: -1
        hard: -1
      nofile:
        soft: 65536
        hard: 65536
    volumes:
      - opensearch-data1:/usr/share/opensearch/data
    ports:
      - "9200:9200"
      - "9600:9600"
    networks:
      - rag-network
    healthcheck:
      test: ["CMD-SHELL", "curl -sf http://localhost:9200/_cluster/health || exit 1"]
      interval: 15s
      timeout: 10s
      retries: 8

  opensearch-dashboards:
    image: opensearchproject/opensearch-dashboards:3.0.0
    container_name: opensearch-dashboards
    ports:
      - "5601:5601"
    environment:
      OPENSEARCH_HOSTS: '["http://opensearch-node1:9200"]'
      DISABLE_SECURITY_DASHBOARDS_PLUGIN: "true"
    depends_on:
      opensearch-node1:
        condition: service_healthy
    networks:
      - rag-network

volumes:
  opensearch-data1:

networks:
  rag-network:
    driver: bridge
```

### Passo B — Subir a Stack

```bash
# No diretório ~/mba-rag/infra:
docker compose up -d

# Aguarda healthcheck (≈ 60-90s)
docker compose ps

# Verifica resposta local
curl http://localhost:9200
# Esperado: {"cluster_name": "opensearch-cluster-rag", ...}
```

### Passo C — Configurar ngrok no Docker Desktop

O **plugin ngrok** do Docker Desktop cria um túnel HTTPS seguro da porta 9200 da sua máquina para uma URL pública acessível de qualquer lugar (inclusive do Colab).

```
1. Abra o Docker Desktop
2. Clique em Extensions (ícone de puzzle, painel esquerdo)
3. Busque "ngrok" e clique em Install
4. Após instalar, clique em Extensions → ngrok
5. Na primeira vez: clique em "Connect your account" e cole seu
   Authtoken (obtido em https://dashboard.ngrok.com/get-started/your-authtoken)
6. Em "Start a tunnel":
   - Port: 9200
   - Protocol: HTTP
   - Clique em "Start Tunnel"
7. Copie a URL gerada (ex: https://a1b2c3d4.ngrok-free.app)
   ↑ Esta é a sua OPENSEARCH_PUBLIC_URL para o próximo passo
```

> **⚠️ Atenção:** Na conta gratuita do ngrok, a URL muda a cada vez que o túnel é reiniciado. Anote a URL antes de executar o Colab.

> **💡 Alternativa CLI:** Se preferir usar o ngrok via linha de comando (sem o Docker Desktop plugin), consulte a seção 3 do Roteiro de Instalação.

---
## 🌐 PARTE 2 — CONEXÃO DO COLAB AO OPENSEARCH LOCAL

A partir daqui, execute as células **no Google Colab**.

## 📦 CÉLULA 1 — Instalar Dependências e Configurar a URL do ngrok

**O que faz:** Instala o cliente Python do OpenSearch e configura a URL pública gerada pelo ngrok.

**Por que:** O Google Colab não consegue acessar `localhost:9200` da sua máquina diretamente. O ngrok cria uma ponte HTTPS entre o Colab e o OpenSearch local.

In [ ]:
# Célula 1: Instalar dependências
!pip install opensearch-py==2.7.1 requests --quiet

import os
import requests
import json

# ─────────────────────────────────────────────────────────────
# ⚙️  CONFIGURAÇÃO: cole aqui a URL pública gerada pelo ngrok
# Exemplo: 'https://a1b2c3d4.ngrok-free.app'
# ─────────────────────────────────────────────────────────────

# Opção 1: Definir diretamente (apenas para testes rápidos)
OPENSEARCH_PUBLIC_URL = 'https://SEU-SUBDOMINIO.ngrok-free.app'  # ← SUBSTITUA AQUI

# Opção 2: Ler do cofre de Segredos do Colab (recomendado)
try:
    from google.colab import userdata
    url_secreta = userdata.get('OPENSEARCH_NGROK_URL')
    if url_secreta:
        OPENSEARCH_PUBLIC_URL = url_secreta
        print('✅ URL carregada dos Segredos do Colab')
except Exception:
    pass

# Remove barra final se houver
OPENSEARCH_PUBLIC_URL = OPENSEARCH_PUBLIC_URL.rstrip('/')

print(f'🌐 OpenSearch URL configurada: {OPENSEARCH_PUBLIC_URL}')

# Validação mínima do formato
if 'SEU-SUBDOMINIO' in OPENSEARCH_PUBLIC_URL:
    print('\n⛔ ATENÇÃO: substitua a URL de exemplo pela URL real do ngrok!')
    print('   Consulte o Passo C da Parte 1 deste notebook.')
else:
    print('✅ Formato da URL validado. Pronto para conectar.')

### 📤 Output Esperado:
```
✅ URL carregada dos Segredos do Colab
🌐 OpenSearch URL configurada: https://a1b2c3d4.ngrok-free.app
✅ Formato da URL validado. Pronto para conectar.
```

### 🔧 Troubleshooting:
| Situação | Ação |
|----------|------|
| `'SEU-SUBDOMINIO' in URL` | Edite a variável `OPENSEARCH_PUBLIC_URL` com a URL real do ngrok |
| URL expirada (ngrok gratuito) | Reinicie o túnel no Docker Desktop e copie a nova URL |

## 📦 CÉLULA 2 — Teste de Conectividade HTTP

**O que faz:** Verifica se o Colab consegue alcançar o OpenSearch via túnel ngrok com uma requisição HTTP direta.

**Por que:** Antes de usar o cliente Python, confirmamos que o túnel está ativo e o OpenSearch está respondendo.

In [ ]:
# Célula 2: Teste de conectividade HTTP via ngrok
import requests
import json
import time

# O ngrok gratuito exige o header 'ngrok-skip-browser-warning'
# para evitar a página de aviso interstitial
HEADERS_NGROK = {
    'ngrok-skip-browser-warning': 'true',
    'Content-Type': 'application/json'
}

print('🔌 TESTE DE CONECTIVIDADE — Colab → ngrok → OpenSearch Local')
print('=' * 65)

def testar_conexao(url, tentativas=5, intervalo=5):
    """Testa a conexão com retry automático."""
    for i in range(1, tentativas + 1):
        try:
            r = requests.get(url, headers=HEADERS_NGROK, timeout=10)
            if r.status_code == 200:
                return r.json()
            else:
                print(f'  Tentativa {i}/{tentativas}: HTTP {r.status_code}')
        except requests.exceptions.ConnectionError as e:
            print(f'  Tentativa {i}/{tentativas}: Conexão recusada — {str(e)[:60]}')
        except requests.exceptions.Timeout:
            print(f'  Tentativa {i}/{tentativas}: Timeout (10s)')
        if i < tentativas:
            time.sleep(intervalo)
    return None

info = testar_conexao(OPENSEARCH_PUBLIC_URL)

if info:
    print('\n✅ CONEXÃO ESTABELECIDA!')
    print(f'   Cluster:  {info.get("cluster_name", "N/A")}')
    print(f'   Versão:   {info.get("version", {}).get("number", "N/A")}')
    print(f'   Tagline:  {info.get("tagline", "N/A")}')

    # Verifica saúde do cluster
    health_url = f'{OPENSEARCH_PUBLIC_URL}/_cluster/health'
    health = requests.get(health_url, headers=HEADERS_NGROK).json()
    emoji = {'green': '🟢', 'yellow': '🟡', 'red': '🔴'}.get(health['status'], '⚪')
    print(f'\n{emoji} Cluster status: {health["status"].upper()}')
    print(f'   Nós:          {health["number_of_nodes"]}')
    print(f'   Shards ativos: {health["active_shards"]}')
    if health['status'] == 'yellow':
        print('   ℹ️  YELLOW normal em single-node dev (réplicas não alocadas)')
else:
    print('\n❌ Falha na conexão após 5 tentativas.')
    print('\n🔍 DIAGNÓSTICO — Verifique:')
    print('  1. Docker Desktop está aberto e em execução?')
    print('  2. Contêiner opensearch-node1 está UP? → docker ps')
    print('  3. Túnel ngrok está ativo? → Extensions → ngrok no Docker Desktop')
    print('  4. A URL do ngrok está correta na Célula 1?')
    print('  5. O OpenSearch está saudável? → curl http://localhost:9200 (na sua máquina)')

### 📤 Output Esperado:
```
🔌 TESTE DE CONECTIVIDADE — Colab → ngrok → OpenSearch Local
=================================================================

✅ CONEXÃO ESTABELECIDA!
   Cluster:  opensearch-cluster-rag
   Versão:   3.0.0
   Tagline:  The OpenSearch Project: https://opensearch.org/

🟡 Cluster status: YELLOW
   Nós:           1
   Shards ativos: 0
   ℹ️  YELLOW normal em single-node dev
```

### 🔧 Troubleshooting:
| Erro | Causa | Solução |
|------|-------|--------|
| `Connection refused` | Túnel ngrok não está ativo | Inicie o túnel no Docker Desktop → Extensions → ngrok |
| `HTTP 502 Bad Gateway` | OpenSearch não iniciou ainda | Aguarde 90s e tente novamente; verifique `docker ps` |
| `HTTP 404 Not Found` | URL ngrok errada | Copie novamente a URL do painel ngrok no Docker Desktop |
| Página HTML de aviso (ngrok) | Header `ngrok-skip-browser-warning` ausente | Verifique se `HEADERS_NGROK` está sendo passado |

## 📦 CÉLULA 3 — Criar Cliente Python com opensearch-py

**O que faz:** Instancia o cliente oficial `opensearch-py` apontando para a URL pública do ngrok.

**Por que:** O cliente abstrai as chamadas HTTP, gerencia retries, serialização JSON e será usado em todos os labs subsequentes.

In [ ]:
# Célula 3: Criar cliente opensearch-py via URL ngrok
from opensearchpy import OpenSearch
from urllib.parse import urlparse

# Extrai host e porta da URL pública do ngrok
parsed = urlparse(OPENSEARCH_PUBLIC_URL)
ngrok_host = parsed.hostname
# ngrok usa porta 443 (HTTPS) ou 80 (HTTP)
ngrok_port = parsed.port or (443 if parsed.scheme == 'https' else 80)
usa_ssl    = parsed.scheme == 'https'

print(f'🔧 Configurando cliente opensearch-py:')
print(f'   Host:   {ngrok_host}')
print(f'   Porta:  {ngrok_port}')
print(f'   SSL:    {usa_ssl}')

client = OpenSearch(
    hosts=[{'host': ngrok_host, 'port': ngrok_port}],
    http_compress=True,
    use_ssl=usa_ssl,
    verify_certs=False,          # ngrok usa certificado autoassinado
    ssl_assert_hostname=False,
    ssl_show_warn=False,
    http_auth=None,              # Segurança desabilitada no compose de dev
    headers={'ngrok-skip-browser-warning': 'true'}  # Necessário para ngrok gratuito
)

# Teste de conexão via cliente Python
try:
    info = client.info()
    print(f'\n✅ Cliente conectado ao OpenSearch {info["version"]["number"]}')
    print(f'   Cluster: {info["cluster_name"]}')
except Exception as e:
    print(f'\n❌ Erro ao conectar via cliente: {e}')
    print('   Verifique a URL do ngrok e o status do OpenSearch.')

## 📦 CÉLULA 4 — Criar Índice Jurídico

**O que faz:** Cria o índice `documentos_juridicos_aula1` com mapeamento especializado para o domínio jurídico.

**Por que:** O mapeamento define como cada campo é tokenizado e indexado. O `juridico_analyzer` normaliza acentos (`réu → reu`) e remove stopwords, melhorando a precisão das buscas.

In [ ]:
# Célula 4: Criar índice de documentos jurídicos
INDEX_NAME = 'documentos_juridicos_aula1'

# Remove índice anterior se existir (para recriar limpo)
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f'🗑️  Índice existente "{INDEX_NAME}" removido')

mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "analysis": {
            "analyzer": {
                "juridico_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase",
                        "asciifolding",  # réu → reu, ação → acao
                        "stop"
                    ]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "titulo":          {"type": "text",    "analyzer": "juridico_analyzer",
                                "fields": {"keyword": {"type": "keyword"}}},
            "conteudo":        {"type": "text",    "analyzer": "juridico_analyzer"},
            "tipo_documento":  {"type": "keyword"},
            "tribunal":        {"type": "keyword"},
            "numero_processo": {"type": "keyword"},
            "data_julgamento": {"type": "date"},
            "relator":         {"type": "keyword"}
        }
    }
}

resp = client.indices.create(index=INDEX_NAME, body=mapping)
if resp.get('acknowledged'):
    print(f'✅ Índice "{INDEX_NAME}" criado no OpenSearch local!')
    print('   Campos: titulo · conteudo · tipo_documento · tribunal')
    print('           numero_processo · data_julgamento · relator')
else:
    print('❌ Falha ao criar índice:', resp)

## 📦 CÉLULA 5 — Inserir Documentos Jurídicos de Exemplo

**O que faz:** Indexa 5 documentos jurídicos fictícios enviados do Colab para o OpenSearch local via túnel ngrok.

**Por que:** Demonstra que a pipeline de escrita remota funciona — os dados ficam na sua máquina, não no Colab.

In [ ]:
# Célula 5: Indexar documentos jurídicos
import time

documentos = [
    {
        "titulo": "Acórdão — Peculato Doloso — Servidor Público da Saúde",
        "conteudo": (
            "EMENTA: PENAL. PECULATO DOLOSO. SERVIDOR PÚBLICO. DESVIO DE VERBAS DO SUS. "
            "MATERIALIDADE E AUTORIA COMPROVADAS. O réu, servidor do Ministério da Saúde, "
            "desviou R$ 2.500.000,00 destinados a equipamentos médicos, configurando o "
            "crime de peculato doloso (art. 312 CP). Recurso desprovido."
        ),
        "tipo_documento": "acordao",
        "tribunal": "STJ",
        "numero_processo": "REsp 1.987.654/SP",
        "data_julgamento": "2024-03-15",
        "relator": "Min. João Silva"
    },
    {
        "titulo": "Habeas Corpus — Prisão Preventiva — Tráfico de Drogas",
        "conteudo": (
            "EMENTA: HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO PREVENTIVA SEM "
            "FUNDAMENTAÇÃO IDÔNEA. EXCESSO DE PRAZO. CONSTRANGIMENTO ILEGAL. ORDEM CONCEDIDA. "
            "Meras referências abstratas à gravidade do delito não bastam para justificar "
            "a custódia cautelar. Medidas alternativas impostas."
        ),
        "tipo_documento": "acordao",
        "tribunal": "STF",
        "numero_processo": "HC 198.432/RJ",
        "data_julgamento": "2024-06-20",
        "relator": "Min. Maria Santos"
    },
    {
        "titulo": "Boletim de Ocorrência — Furto Qualificado — Escalada",
        "conteudo": (
            "BO Nº 2024/0045678. Furto Qualificado por escalada. Vítima retornou ao domicílio "
            "e encontrou janela arrombada. Bens subtraídos: 1 notebook, 2 celulares, "
            "R$1.200,00. Impressões digitais coletadas. Câmera de segurança registrou suspeito "
            "encapuzado. Inquérito instaurado pela delegada responsável."
        ),
        "tipo_documento": "boletim_ocorrencia",
        "tribunal": "PCSP",
        "numero_processo": "BO 2024/0045678",
        "data_julgamento": "2024-11-12",
        "relator": "Del. Carlos Mendes"
    },
    {
        "titulo": "Laudo Pericial — Homicídio — Análise Balística",
        "conteudo": (
            "LAUDO Nº 2024/LP-00234. Análise balística: 3 estojos calibre 9mm da mesma arma "
            "(análise de estrias). Trajetória compatível com vítima ajoelhada. "
            "GSR (Gunshot Residue) encontrado nas mãos do suspeito. "
            "Conclusão: elementos compatíveis com homicídio qualificado por motivo torpe."
        ),
        "tipo_documento": "laudo_pericial",
        "tribunal": "IML-SP",
        "numero_processo": "IP 2024/00789",
        "data_julgamento": "2024-09-08",
        "relator": "Perita Dra. Ana Lima"
    },
    {
        "titulo": "Sentença — Absolvição por Legítima Defesa — Homicídio",
        "conteudo": (
            "SENTENÇA ABSOLUTÓRIA. Processo: 0001234-56.2024.8.26.0001. "
            "O réu agiu em legítima defesa própria, com uso moderado dos meios "
            "necessários para repelir injusta agressão atual (vítima avançou com faca). "
            "Testemunhas oculares confirmaram a versão defensiva. "
            "Fundamento: art. 25 CP c/c art. 386 VI CPP. Réu absolvido."
        ),
        "tipo_documento": "sentenca",
        "tribunal": "TJ-SP",
        "numero_processo": "0001234-56.2024.8.26.0001",
        "data_julgamento": "2024-12-01",
        "relator": "Juíza Dra. Paula Costa"
    }
]

print(f'📥 Indexando {len(documentos)} documentos no OpenSearch local via ngrok...')
print(f'   Destino: {OPENSEARCH_PUBLIC_URL}/{INDEX_NAME}')
print('=' * 65)

sucesso = 0
for i, doc in enumerate(documentos, 1):
    try:
        resp = client.index(
            index=INDEX_NAME,
            body=doc,
            id=str(i),
            refresh=True          # Índice atualizado imediatamente
        )
        ok = resp['result'] in ['created', 'updated']
        print(f'  {"✅" if ok else "❌"} [{i}] {doc["titulo"][:55]}...')
        if ok:
            sucesso += 1
    except Exception as e:
        print(f'  ❌ [{i}] Erro: {str(e)[:80]}')

total = client.count(index=INDEX_NAME)['count']
print(f'\n✅ {sucesso}/{len(documentos)} documentos indexados | Total no índice: {total}')

## 📦 CÉLULA 6 — Validação de Buscas pelo Colab

**O que faz:** Executa 3 buscas full-text no OpenSearch local a partir do Colab, validando o pipeline completo.

**Por que:** Confirma que a comunicação bidirecional Colab ↔ ngrok ↔ OpenSearch está funcionando para leitura (busca) além de escrita (indexação).

In [ ]:
# Célula 6: Executar buscas de validação via Colab → ngrok → OpenSearch local

def buscar(query_text, filtro_tipo=None, top_k=3):
    """Busca full-text no OpenSearch com filtro opcional por tipo de documento."""
    query = {
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query_text,
                        "fields": ["titulo^2", "conteudo"],
                        "type": "best_fields"
                    }
                }],
                "filter": ([{"term": {"tipo_documento": filtro_tipo}}]
                           if filtro_tipo else [])
            }
        },
        "size": top_k,
        "_source": ["titulo", "tipo_documento", "tribunal", "numero_processo"]
    }
    return client.search(index=INDEX_NAME, body=query)


print('🔍 BUSCAS DE VALIDAÇÃO — Colab → ngrok → OpenSearch Local')
print('=' * 65)

casos_teste = [
    ('peculato servidor público desvio verba', None),
    ('homicídio balística projéteis arma', 'laudo_pericial'),
    ('prisão preventiva fundamentação cautelar', None),
]

todos_ok = True
for query, filtro in casos_teste:
    print(f'\n  🔎 Query: "{query}"' + (f'  [filtro: {filtro}]' if filtro else ''))
    try:
        hits = buscar(query, filtro_tipo=filtro)['hits']['hits']
        if hits:
            for h in hits:
                src = h['_source']
                print(f'     📄 [{h["_score"]:.2f}] {src["titulo"][:55]}...')
                print(f'          {src["tipo_documento"]} | {src["tribunal"]}')
        else:
            print('     ⚠️  Nenhum resultado retornado')
            todos_ok = False
    except Exception as e:
        print(f'     ❌ Erro na busca: {e}')
        todos_ok = False

print('\n' + '=' * 65)
if todos_ok:
    print('✅ TODAS AS BUSCAS RETORNARAM RESULTADOS RELEVANTES!')
    print('   Pipeline completo validado: Colab → ngrok → OpenSearch → resposta')
else:
    print('⚠️  Algumas buscas não retornaram resultados. Verifique a indexação (Célula 5).')

### 📤 Output Esperado:
```
🔍 BUSCAS DE VALIDAÇÃO — Colab → ngrok → OpenSearch Local
=================================================================

  🔎 Query: "peculato servidor público desvio verba"
     📄 [8.45] Acórdão — Peculato Doloso — Servidor Público...
          acordao | STJ

  🔎 Query: "homicídio balística projéteis arma"  [filtro: laudo_pericial]
     📄 [6.21] Laudo Pericial — Homicídio — Análise Balística...
          laudo_pericial | IML-SP

  🔎 Query: "prisão preventiva fundamentação cautelar"
     📄 [9.12] Habeas Corpus — Prisão Preventiva — Tráfico...
          acordao | STF

=================================================================
✅ TODAS AS BUSCAS RETORNARAM RESULTADOS RELEVANTES!
   Pipeline completo validado: Colab → ngrok → OpenSearch → resposta
```

## 📦 CÉLULA 7 — Script de Validação Completa (Checklist Automatizado)

**O que faz:** Executa automaticamente todos os 8 itens do checklist de validação do Lab 1.

**Por que:** Fornece uma verificação objetiva e reproduzível do ambiente para fins de avaliação.

In [ ]:
# Célula 7: Script de validação completa — Checklist automatizado
from datetime import datetime

checklist = []

def check(nome, fn):
    try:
        resultado = fn()
        checklist.append(('✅', nome, str(resultado)[:60] if resultado else 'OK'))
    except Exception as e:
        checklist.append(('❌', nome, str(e)[:60]))

print(f'📋 CHECKLIST DO LAB 1 — {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print('=' * 65)

# 1. Biblioteca opensearch-py importada
def c1():
    from opensearchpy import OpenSearch
    return 'opensearch-py importado'
check('opensearch-py instalado', c1)

# 2. URL ngrok configurada
def c2():
    assert 'ngrok' in OPENSEARCH_PUBLIC_URL or OPENSEARCH_PUBLIC_URL.startswith('https://'), \
        'URL não parece ser ngrok'
    return OPENSEARCH_PUBLIC_URL
check('URL ngrok configurada', c2)

# 3. OpenSearch acessível via ngrok
def c3():
    r = requests.get(OPENSEARCH_PUBLIC_URL, headers=HEADERS_NGROK, timeout=10)
    assert r.status_code == 200
    return f'versão {r.json()["version"]["number"]}'
check('OpenSearch acessível via ngrok', c3)

# 4. Cliente Python conectado
def c4():
    info = client.info()
    return f'cluster {info["cluster_name"]}'
check('Cliente opensearch-py conectado', c4)

# 5. Cluster saudável (green ou yellow)
def c5():
    h = requests.get(f'{OPENSEARCH_PUBLIC_URL}/_cluster/health',
                     headers=HEADERS_NGROK).json()
    assert h['status'] in ['green', 'yellow'], f'Status: {h["status"]}'
    return f'status {h["status"]}'
check('Cluster saudável (green/yellow)', c5)

# 6. Índice criado
def c6():
    assert client.indices.exists(index=INDEX_NAME), 'Índice não encontrado'
    return f'índice {INDEX_NAME} existe'
check(f'Índice "{INDEX_NAME}" criado', c6)

# 7. Documentos indexados
def c7():
    cnt = client.count(index=INDEX_NAME)['count']
    assert cnt >= 5, f'Apenas {cnt} documentos'
    return f'{cnt} documentos'
check('5+ documentos indexados', c7)

# 8. Busca retorna resultados relevantes
def c8():
    hits = buscar('peculato servidor público')['hits']['hits']
    assert len(hits) > 0, 'Nenhum resultado'
    top = hits[0]['_source']['titulo']
    return f'top: {top[:40]}'
check('Busca retorna resultados', c8)

# Exibe checklist
print()
for status, nome, detalhe in checklist:
    print(f'  {status} {nome}')
    if status == '❌':
        print(f'     └─ {detalhe}')

ok = sum(1 for s, _, _ in checklist if s == '✅')
total = len(checklist)
print(f'\n  Pontuação: {ok}/{total}')

if ok == total:
    print('\n🎉 LAB 1 COMPLETAMENTE VALIDADO!')
    print('   Pipeline Colab → ngrok → OpenSearch Local funcionando!')
else:
    print(f'\n⚠️  {total-ok} item(ns) com falha. Consulte o troubleshooting acima.')

## ✅ Checklist de Validação do Lab 1

| # | Item | Verificação |
|---|------|------------|
| 1 | Docker Desktop instalado e em execução | `docker ps` na sua máquina |
| 2 | Stack OpenSearch UP via `docker compose up -d` | Contêineres green no Docker Desktop |
| 3 | Plugin ngrok instalado e autenticado | Extensions → ngrok no Docker Desktop |
| 4 | Túnel ngrok ativo na porta 9200 | URL pública exibida no painel ngrok |
| 5 | Colab conecta ao OpenSearch via URL ngrok | Célula 2: ✅ CONEXÃO ESTABELECIDA |
| 6 | Índice `documentos_juridicos_aula1` criado | Célula 4: acknowledged=True |
| 7 | 5 documentos indexados com sucesso | Célula 5: 5/5 ✅ |
| 8 | 3 buscas retornando resultados relevantes | Células 6-7: todas ✅ |

**Pontuação:** 8/8 = Lab 1 completo ✅

### 🔧 Troubleshooting Geral:
| Erro | Causa | Solução |
|------|-------|--------|
| `Connection refused` no Colab | Túnel ngrok parado | Reinicie o túnel em Extensions → ngrok |
| `HTTP 502` | OpenSearch não iniciou | `docker compose ps` na sua máquina; aguarde 90s |
| `HTTP 401` | Autenticação habilitada | Verifique `plugins.security.disabled=true` no compose |
| URL mudou (ngrok gratuito) | Túnel reiniciado | Atualize `OPENSEARCH_PUBLIC_URL` na Célula 1 |
| `vm.max_map_count` baixo | Parâmetro do kernel | Veja seção 1.4 do Roteiro de Instalação |
| Dashboard inacessível | Porta 5601 não exposta | Crie segundo túnel ngrok para a porta 5601 |